In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/alfathterry/telco-customer-churn-11-1-3/telco.csv


In [2]:
%load_ext cuml.accel
import sklearn

In [3]:
# General purpose modules import
import time
from copy import deepcopy
import warnings

# Data handling and visualization modules
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import math

# Skikit-learn preprocessing and evaluation modules
from sklearn.feature_selection import SelectFromModel
from sklearn.model_selection import ShuffleSplit
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.model_selection import KFold
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.compose import make_column_selector
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import FunctionTransformer, MinMaxScaler, RobustScaler
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder, LabelEncoder
from sklearn.preprocessing import TargetEncoder, KBinsDiscretizer, StandardScaler
from sklearn.metrics import balanced_accuracy_score, roc_auc_score, accuracy_score

# Skikit-learn ML modules
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.ensemble import HistGradientBoostingClassifier, AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import SGDClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.utils.class_weight import compute_class_weight

# Further ML modules
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
#SHAP
import shap
from sklearn.inspection import permutation_importance
from sklearn.inspection import PartialDependenceDisplay

In [4]:
# gpu import
# NN model
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

# new cuML speedup 
import cudf
from cuml.linear_model import LogisticRegression as cuLR
from cuml.ensemble import RandomForestClassifier as cuRF
from cuml.svm import SVC as cuSVC
from cuml.neighbors import KNeighborsClassifier as cuKNN

In [5]:
import warnings
from pandas.errors import PerformanceWarning

warnings.filterwarnings("ignore", category=PerformanceWarning)

In [6]:
df = pd.read_csv('/kaggle/input/datasets/alfathterry/telco-customer-churn-11-1-3/telco.csv')

train_df, test_df = train_test_split(df, stratify = df['Churn Label'],test_size = 0.2,  random_state = 1)

In [ ]:
FEATURES = ['Customer ID', 'Gender', 'Age', 'Under 30', 'Senior Citizen', 'Married', 'Dependents', 
            'Number of Dependents', 'Country', 'State', 'City', 'Zip Code', 'Latitude', 'Longitude',
            'Population', 'Quarter', 'Referred a Friend', 'Number of Referrals', 'Tenure in Months',
            'Offer', 'Phone Service', 'Avg Monthly Long Distance Charges', 'Multiple Lines', 
            'Internet Service', 'Internet Type', 'Avg Monthly GB Download', 'Online Security', 
            'Online Backup', 'Device Protection Plan', 'Premium Tech Support', 'Streaming TV',
            'Streaming Movies', 'Streaming Music', 'Unlimited Data', 'Contract', 'Paperless Billing',
            'Payment Method', 'Monthly Charge', 'Total Charges', 'Total Refunds',
            'Total Extra Data Charges', 'Total Long Distance Charges', 'Total Revenue', 
            'Satisfaction Score', 'Customer Status', 'Churn Score', 'CLTV', 
            'Churn Category', 'Churn Reason']

TARGET = 'Churn Label'

mappings = {'No': 0, 'Yes': 1}
reverse_mappings = {0 : 'No', 1: 'Yes'}

train_df['Churn Label'] = train_df['Churn Label'].map(mappings)
test_df['Churn Label'] = test_df['Churn Label'].map(mappings)

train_df = train_df.drop('Customer ID',axis = 1, errors = 'ignore')
test_df = test_df.drop('Customer ID', axis = 1 , errors = 'ignore')

train_df['Offer'] = train_df['Offer'].fillna('None')
test_df['Offer'] = test_df['Offer'].fillna('None')

train_df['Internet Type'] = train_df['Internet Type'].fillna('No')
test_df['Internet Type'] = test_df['Internet Type'].fillna('No')

train_df['Zip Code'] = train_df['Zip Code'].astype(str)
test_df['Zip Code'] = test_df['Zip Code'].astype(str)

train_df = train_df.drop(['Under 30'], axis = 1, errors = 'ignore')
test_df = test_df.drop(['Under 30'], axis = 1 , errors = 'ignore')

train_df = train_df.drop(['Churn Score', 'CLTV', 'Churn Category', 'Churn Reason'], axis = 1, errors = 'ignore')
test_df = test_df.drop(['Churn Score', 'CLTV', 'Churn Category', 'Churn Reason'], axis = 1 , errors = 'ignore')

train_df = train_df.drop(['Country','State','Quarter'], axis = 1, errors = 'ignore')
test_df = test_df.drop(['Country','State','Quarter'], axis = 1 , errors = 'ignore')

train_df = train_df.drop(['Customer Status','Satisfaction Score'], axis = 1, errors = 'ignore')
test_df = test_df.drop(['Customer Status','Satisfaction Score'], axis = 1 , errors = 'ignore')

num_cols = ['Age', 'Number of Dependents', 'Latitude', 'Longitude', 'Population', 
            'Number of Referrals', 'Tenure in Months', 'Avg Monthly Long Distance Charges', 
            'Avg Monthly GB Download', 'Monthly Charge', 'Total Charges', 'Total Refunds', 
            'Total Extra Data Charges', 'Total Long Distance Charges', 'Total Revenue']

cat_cols = ['Gender', 'Senior Citizen', 'Married', 'Dependents', 'City', 
            'Referred a Friend', 'Offer', 'Phone Service', 'Multiple Lines', 'Internet Service',
            'Internet Type', 'Online Security', 'Online Backup', 'Device Protection Plan', 
            'Premium Tech Support', 'Streaming TV', 'Streaming Movies', 'Streaming Music', 
            'Unlimited Data', 'Contract', 'Paperless Billing', 'Payment Method','Zip Code']


In [8]:
def feature_engineering(
    X_train,
    X_val,
    y_train,
    num_cols = num_cols,
    cat_cols = cat_cols
):
    X_train = X_train.copy()
    X_val = X_val.copy()
    FE = []
    ###_Latitude_X_Longitude FE###
    
    # Round coordinates
    for df in [X_train, X_val]:
        df["Latitude_2f"] = df["Latitude"].round(2)
        df["Longitude_2f"] = df["Longitude"].round(2)
    
        # Geographic bucket
        df["lat_lon_grid"] = (df["Latitude_2f"].astype(str) + "_" + df["Longitude_2f"].astype(str))
    
        # Numeric interactions
        df["lat_x_lon"] = (df["Latitude_2f"] * df["Longitude_2f"])
    
        df["lat_plus_lon"] = (df["Latitude_2f"] + df["Longitude_2f"])
    
        df["lat_minus_lon"] = (df["Latitude_2f"] - df["Longitude_2f"])
    
    # Distance from train center 
    lat_center = X_train["Latitude"].median()
    lon_center = X_train["Longitude"].median()
    
    for df in [X_train, X_val]:
        df["dist_center"] = np.sqrt(
            (df["Latitude"] - lat_center) ** 2 +
            (df["Longitude"] - lon_center) ** 2
        )
    # =========================
    # Population Interactions
    # =========================
    
    for df in [X_train, X_val]:
    
        df["log_population"] = np.log1p(df["Population"])
    
        # Population × Coordinates
        df["pop_x_lat"] = (df["Population"] * df["Latitude"])
    
        df["pop_x_lon"] = (df["Population"] * df["Longitude"])
    
        # Log-population × Coordinates
        df["logpop_x_lat"] = (df["log_population"] * df["Latitude"])
    
        df["logpop_x_lon"] = (df["log_population"] * df["Longitude"])
    
        # Population × Rounded Coordinates
        df["pop_x_lat2f"] = (df["Population"] * df["Latitude_2f"])
    
        df["pop_x_lon2f"] = (df["Population"] * df["Longitude_2f"])
    
        # Population relative to center
        df["pop_per_dist"] = (df["Population"] / (df["dist_center"] + 1e-6))
    
        # Population + Geo Bucket (for CatBoost)
        df["pop_geo"] = ((df["Population"] // 1000).astype(int).astype(str) + "_" + df["lat_lon_grid"])
    
    new_features = [
        "Latitude_2f","Longitude_2f",
        "lat_lon_grid","lat_x_lon",
        "lat_plus_lon","lat_minus_lon",
        "dist_center",
        "log_population",
        "pop_x_lat","pop_x_lon",
        "logpop_x_lat","logpop_x_lon",
        "pop_x_lat2f","pop_x_lon2f",
        "pop_per_dist","pop_geo",
    ]
    # orinal enc
    cat_features_new = [
        "lat_lon_grid",
        "pop_geo",
    ]
    
    oe = OrdinalEncoder(handle_unknown="use_encoded_value",unknown_value=-1)
    
    X_train[cat_features_new] = oe.fit_transform(
        X_train[cat_features_new]
    )
    
    X_val[cat_features_new] = oe.transform(
        X_val[cat_features_new]
    )
    
    FE += new_features
    ### Referrals and Dependents
    
    new_features = [
    'referrals_x_dependents',
    'referrals_plus_dependents',
    'referrals_minus_dependents',
    'referrals_per_dependent',
    'has_referrals',
    'has_dependents',
    'has_referrals_and_dependents'
]
    for df in [X_train, X_val]:
        # Basic interactions
        df['referrals_x_dependents'] = (
            df['Number of Referrals'] * df['Number of Dependents']
        )
    
        df['referrals_plus_dependents'] = (
            df['Number of Referrals'] + df['Number of Dependents']
        )
    
        df['referrals_minus_dependents'] = (
            df['Number of Referrals'] - df['Number of Dependents']
        )
    
        df['referrals_per_dependent'] = (
            df['Number of Referrals'] / (df['Number of Dependents'] + 1)
        )
    
        # is Referrals interactions
        df['has_referrals'] = (
            df['Number of Referrals'] > 0
        ).astype(int)
    
        df['has_dependents'] = (
            df['Number of Dependents'] > 0
        ).astype(int)
    
        df['has_referrals_and_dependents'] = (
            (df['Number of Referrals'] > 0) &
            (df['Number of Dependents'] > 0)
        ).astype(int)
        
    FE += new_features
    
    for df in [X_train, X_val]:
    
        # Revenue / tenure interactions
        df['Revenue_per_Month'] = (
            df['Total Revenue'] / (df['Tenure in Months'] + 1)
        )
    
        df['Charges_per_Month'] = (
            df['Total Charges'] / (df['Tenure in Months'] + 1)
        )
    
        df['Refund_per_Month'] = (
            df['Total Refunds'] / (df['Tenure in Months'] + 1)
        )
    
        # Actual vs expected spending
        df['Expected_Charges'] = (
            df['Monthly Charge'] * df['Tenure in Months']
        )
    
        df['Charge_Gap'] = (
            df['Total Charges'] - df['Expected_Charges']
        )
    
        df['Charge_Ratio'] = (
            df['Total Charges'] / (df['Expected_Charges'] + 1)
        )
    
        # Refund-related interactions
        df['Refund_Rate'] = (
            df['Total Refunds'] / (df['Total Charges'] + 1)
        )
    
        df['Net_Revenue_Ratio'] = (
            df['Total Revenue'] / (df['Total Charges'] + 1)
        )
    
        df['Refunds_x_Tenure'] = (
            df['Total Refunds'] * np.log1p(df['Tenure in Months'])
        )
    
        # Long-distance behavior
        df['LD_Ratio'] = (
            df['Total Long Distance Charges'] / (df['Total Charges'] + 1)
        )
    
        df['AvgLD_x_Tenure'] = (
            df['Avg Monthly Long Distance Charges']
            * df['Tenure in Months']
        )
    
        df['LD_Gap'] = (
            df['Total Long Distance Charges']
            - df['Avg Monthly Long Distance Charges']
            * df['Tenure in Months']
        )
    
        # Data usage interactions
        df['GB_per_Dollar'] = (
            df['Avg Monthly GB Download']
            / (df['Monthly Charge'] + 1)
        )
    
        df['GB_x_MonthlyCharge'] = (
            df['Avg Monthly GB Download']
            * df['Monthly Charge']
        )
    
        # Extra-data spending
        df['ExtraData_Rate'] = (
            df['Total Extra Data Charges']
            / (df['Total Revenue'] + 1)
        )
    
        df['ExtraData_per_Month'] = (
            df['Total Extra Data Charges']
            / (df['Tenure in Months'] + 1)
        )
    
        # Revenue composition
        df['LD_Revenue_Share'] = (
            df['Total Long Distance Charges']
            / (df['Total Revenue'] + 1)
        )
    
        df['ExtraData_Revenue_Share'] = (
            df['Total Extra Data Charges']
            / (df['Total Revenue'] + 1)
        )
    
        df['Refund_Revenue_Share'] = (
            df['Total Refunds']
            / (df['Total Revenue'] + 1)
        )
    
        # Polynomial interactions
        df['Tenure_x_MonthlyCharge'] = (
            df['Tenure in Months']
            * df['Monthly Charge']
        )
    
        df['Tenure_x_GB'] = (
            df['Tenure in Months']
            * df['Avg Monthly GB Download']
        )
    
        df['MonthlyCharge_x_GB'] = (
            df['Monthly Charge']
            * df['Avg Monthly GB Download']
        )
    
        df['Refunds_x_MonthlyCharge'] = (
            df['Total Refunds']
            * df['Monthly Charge']
        )

    # Numerical features
    for col in num_cols:
        FE.append(col)

    # Ordinal Encoding
    for col in cat_cols:
        oe = OrdinalEncoder(
            handle_unknown="use_encoded_value",
            unknown_value=-1,
        )

        X_train[f"oe_{col}"] = oe.fit_transform(X_train[[col]])
        X_val[f"oe_{col}"] = oe.transform(X_val[[col]])
        FE.append(f"oe_{col}")

    # Target Encoding
    for col in cat_cols:
        te = TargetEncoder(random_state=42)

        X_train[f"te_{col}"] = te.fit_transform(
            X_train[[col]], y_train
        )
        X_val[f"te_{col}"] = te.transform(
            X_val[[col]]
        )
        FE.append(f"te_{col}")
######
    # Categorical interaction features + Target Encoding
    
    # Pairwise interactions
    interaction_pairs = [
    
        # Contract-focused
        ('Contract', 'Payment Method'),
        ('Contract', 'Internet Type'),
        ('Contract', 'City'),
        ('Contract', 'Dependents'),
        ('Contract', 'Referred a Friend'),
        ('Contract', 'Offer'),
        ('Contract', 'Streaming TV'),
        ('Contract', 'Streaming Movies'),
        ('Contract', 'Streaming Music'),
    
        # City-focused
        ('City', 'Internet Type'),
        ('City', 'Offer'),
        ('City', 'Payment Method'),
    
        # Referral / family
        ('Referred a Friend', 'Dependents'),
        ('Referred a Friend', 'Offer'),
    
        # Internet-focused
        ('Internet Type', 'Offer'),
        ('Internet Type', 'Payment Method'),
    
        # Streaming-focused
        ('Streaming TV', 'Streaming Movies'),
        ('Streaming TV', 'Streaming Music'),
        ('Streaming Movies', 'Streaming Music'),
    
        ('Internet Type', 'Streaming TV'),
        ('Internet Type', 'Streaming Movies'),
        ('Internet Type', 'Streaming Music'),
    
        # Geography
        ('City', 'Dependents'),
        ('City', 'Referred a Friend'),
    ]
    
    interaction_cols = []
    
    for c1, c2 in interaction_pairs:
    
        feat = f'{c1}_{c2}_FE'
    
        X_train[feat] = (
            X_train[c1].astype(str)
            + '__'
            + X_train[c2].astype(str)
        )
    
        X_val[feat] = (
            X_val[c1].astype(str)
            + '__'
            + X_val[c2].astype(str)
        )
    
        interaction_cols.append(feat)
    
    # --------------------------
    # Streaming bundle
    # --------------------------
    stream_cols = [
        'Streaming TV',
        'Streaming Movies',
        'Streaming Music'
    ]
    
    feat = 'Streaming_Bundle_FE'
    
    X_train[feat] = (
        X_train[stream_cols]
        .astype(str)
        .agg('__'.join, axis=1)
    )
    
    X_val[feat] = (
        X_val[stream_cols]
        .astype(str)
        .agg('__'.join, axis=1)
    )
    
    interaction_cols.append(feat)
    
    # --------------------------
    # Streaming count
    # --------------------------
    stream_map = {
        'Yes': 1,
        'No': 0
    }
    
    feat = 'Streaming_Count_FE'
    
    X_train[feat] = (
        X_train[stream_cols]
        .apply(lambda s: s.map(stream_map))
        .fillna(0)
        .sum(axis=1)
        .astype(int)
        .astype(str)
    )
    
    X_val[feat] = (
        X_val[stream_cols]
        .apply(lambda s: s.map(stream_map))
        .fillna(0)
        .sum(axis=1)
        .astype(int)
        .astype(str)
)
    
    interaction_cols.append(feat)
    
    # --------------------------
    # Contract x Streaming Count
    # --------------------------
    feat = 'Contract_StreamCount_FE'
    
    X_train[feat] = (
        X_train['Contract'].astype(str)
        + '__'
        + X_train['Streaming_Count_FE']
    )
    
    X_val[feat] = (
        X_val['Contract'].astype(str)
        + '__'
        + X_val['Streaming_Count_FE']
    )
    
    interaction_cols.append(feat)
    
    # --------------------------
    # City x Streaming Bundle
    # --------------------------
    feat = 'City_StreamingBundle_FE'
    
    X_train[feat] = (
        X_train['City'].astype(str)
        + '__'
        + X_train['Streaming_Bundle_FE']
    )
    
    X_val[feat] = (
        X_val['City'].astype(str)
        + '__'
        + X_val['Streaming_Bundle_FE']
    )
    
    interaction_cols.append(feat)
    
    # --------------------------
    # Triple interactions
    # --------------------------
    triple_features = {
        'Contract_Internet_Offer_FE':
            ['Contract', 'Internet Type', 'Offer'],
    
        'City_Contract_Internet_FE':
            ['City', 'Contract', 'Internet Type'],
    
        'Referral_Dependent_Contract_FE':
            ['Referred a Friend', 'Dependents', 'Contract'],
    
        'Contract_Payment_Internet_FE':
            ['Contract', 'Payment Method', 'Internet Type'],
    
        'City_Offer_Internet_FE':
            ['City', 'Offer', 'Internet Type'],
    }
    
    for feat, cols in triple_features.items():
    
        X_train[feat] = (
            X_train[cols]
            .astype(str)
            .agg('__'.join, axis=1)
        )
    
        X_val[feat] = (
            X_val[cols]
            .astype(str)
            .agg('__'.join, axis=1)
        )
    
        interaction_cols.append(feat)
    
    # ==========================================================
    # Target Encoding for engineered features
    # ==========================================================
    
    for col in interaction_cols:
    
        te = TargetEncoder(random_state=42)
    
        X_train[f"te_{col}"] = te.fit_transform(
            X_train[[col]],
            y_train
        )
    
        X_val[f"te_{col}"] = te.transform(
            X_val[[col]]
        )
    
        FE.append(f"te_{col}")
    
    return X_train, X_val, FE

In [12]:
# train oof and eval linear, random forest, svm and knn 

# =========================================================
# GPU / cuML ACCELERATED PARAMS
# =========================================================
params = {
    # cuML Logistic Regression
    "Linear": {
        "C": 1.0,
        "max_iter": 1000,},
    # cuML RandomForest
    "RandomForest": {
        "n_estimators": 1000,
        "max_depth": 16,
        "random_state": 42,
        "n_streams": 4},
    # cuML SVM
    "SVM": {
        "C": 1.0,
        "kernel": "rbf",
        "probability": True,},
    # cuML KNN
    "KNN": {
        "n_neighbors": 5,}
}
models = {
    # cuML GPU Logistic Regression
    "Linear": cuLR(
        **params["Linear"]),
    # cuML GPU RandomForest
    "RandomForest": cuRF(
        **params["RandomForest"]),
    # cuML GPU SVM
    "SVM": cuSVC(
        **params["SVM"]),
    # cuML GPU KNN
    "KNN": cuKNN(
        **params["KNN"]),
}

N_SPLITS = 3
skf = StratifiedKFold(n_splits=N_SPLITS,shuffle=True,random_state=42)

results = []
X = train_df[num_cols + cat_cols].copy()
y = train_df[TARGET]
test = test_df[num_cols + cat_cols].copy()

for model_name, model in models.items():
    print("=" * 60)
    print(f"TRAINING: {model_name}")
    fold_scores = []
    
    # OOF storage
    oof_preds = np.zeros(len(train_df))
    test_preds = np.zeros(len(test), dtype=np.float32)

    for fold, (train_idx, valid_idx) in enumerate(skf.split(X, y)):
        
        X_train, X_val = X.iloc[train_idx].copy(), X.iloc[valid_idx].copy()
        y_train, y_val = y.iloc[train_idx], y.iloc[valid_idx]

        X_train_prep, X_val_prep, FE = feature_engineering(
            X_train,X_val,y_train,
        )
        
        
        model.fit(X_train_prep[FE],y_train)

        val_preds = model.predict(X_val_prep[FE])
                                    
        # store OOF
        oof_preds[valid_idx] = val_preds

        # AUC SCORE
        score = accuracy_score(y_val, val_preds)
        fold_scores.append(score)
        print(f"Fold {fold + 1}: {score:.5f}")
        
    # =====================================================
    # FINAL SCORE
    # =====================================================

    X_train_prep, test_prep, FE = feature_engineering(
        X,test,y
    )
    model.fit(X_train_prep[FE],y)

    test_preds = model.predict(test_prep[FE])
    test_score = accuracy_score(test_df[TARGET], test_preds)
    
    mean_score = np.mean(fold_scores)
    results.append({
        "model": model_name,
        "acc_mean": mean_score,
        "test_score": test_score
    })
    print(f"{model_name} Mean ACC: {mean_score:.5f}")
    print(f"{model_name} test ACC: {test_score:.5f}")

results_df = pd.DataFrame(results)
results_df = results_df.sort_values(
    by="acc_mean",
    ascending=False
)

print("\nFINAL RESULTS")
print(results_df)

TRAINING: Linear
Fold 1: 0.73589
Fold 2: 0.73536
Fold 3: 0.73482
Linear Mean ACC: 0.73536
Linear test ACC: 0.73314
TRAINING: RandomForest
Fold 1: 0.83600
Fold 2: 0.84931
Fold 3: 0.84239
RandomForest Mean ACC: 0.84256
RandomForest test ACC: 0.85806
TRAINING: SVM
Fold 1: 0.73429
Fold 2: 0.73482
Fold 3: 0.73482
SVM Mean ACC: 0.73465
SVM test ACC: 0.73456
TRAINING: KNN
Fold 1: 0.71619
Fold 2: 0.69489
Fold 3: 0.72151
KNN Mean ACC: 0.71086
KNN test ACC: 0.70901

FINAL RESULTS
          model  acc_mean  test_score
1  RandomForest  0.842563    0.858055
0        Linear  0.735357    0.733144
2           SVM  0.734647    0.734564
3           KNN  0.710863    0.709013


In [13]:
models = {
    "xgb": XGBClassifier(
        n_estimators=2000,
        max_depth=6,
        learning_rate=0.05,
        random_state=42,
        eval_metric="logloss"
    ),

    "lgb": LGBMClassifier(
        n_estimators=2000,
        max_depth=6,
        learning_rate=0.05,
        random_state=42,
        verbose=-1
    ),

    "cat": CatBoostClassifier(
        iterations=2000,
        depth=6,
        learning_rate=0.05,
        random_seed=42,
        verbose=0
    )
}


N_SPLITS = 3
skf = StratifiedKFold(n_splits=N_SPLITS,shuffle=True,random_state=42)

results = []
X = train_df[num_cols + cat_cols].copy()
y = train_df[TARGET]
test = test_df[num_cols + cat_cols].copy()

for model_name, model in models.items():
    print("=" * 60)
    print(f"TRAINING: {model_name}")
    fold_scores = []
    
    # OOF storage
    oof_preds = np.zeros(len(train_df))
    test_preds = np.zeros(len(test), dtype=np.float32)

    for fold, (train_idx, valid_idx) in enumerate(skf.split(X, y)):
        
        X_train, X_val = X.iloc[train_idx].copy(), X.iloc[valid_idx].copy()
        y_train, y_val = y.iloc[train_idx], y.iloc[valid_idx]

        X_train_prep, X_val_prep, FE = feature_engineering(
            X_train,X_val,y_train,
        )
        
        
        model.fit(X_train_prep[FE],y_train)

        val_preds = model.predict(X_val_prep[FE])
                                    
        # store OOF
        oof_preds[valid_idx] = val_preds

        # AUC SCORE
        score = accuracy_score(y_val, val_preds)
        fold_scores.append(score)
        print(f"Fold {fold + 1}: {score:.5f}")
        
    # =====================================================
    # FINAL SCORE
    # =====================================================

    X_train_prep, test_prep, FE = feature_engineering(
        X,test,y
    )
    model.fit(X_train_prep[FE],y)

    test_preds = model.predict(test_prep[FE])
    test_score = accuracy_score(test_df[TARGET], test_preds)
    
    mean_score = np.mean(fold_scores)
    results.append({
        "model": model_name,
        "acc_mean": mean_score,
        "test_score": test_score
    })
    print(f"{model_name} Mean ACC: {mean_score:.5f}")
    print(f"{model_name} test ACC: {test_score:.5f}")

results_df = pd.DataFrame(results)
results_df = results_df.sort_values(
    by="acc_mean",
    ascending=False
)

print("\nFINAL RESULTS")
print(results_df)

TRAINING: xgb
Fold 1: 0.83280
Fold 2: 0.84452
Fold 3: 0.83866
xgb Mean ACC: 0.83866
xgb test ACC: 0.85522
TRAINING: lgb
Fold 1: 0.83440
Fold 2: 0.84345
Fold 3: 0.83759
lgb Mean ACC: 0.83848
lgb test ACC: 0.85593
TRAINING: cat
Fold 1: 0.84292
Fold 2: 0.84984
Fold 3: 0.84611
cat Mean ACC: 0.84629
cat test ACC: 0.86018

FINAL RESULTS
  model  acc_mean  test_score
2   cat  0.846290    0.860185
0   xgb  0.838658    0.855216
1   lgb  0.838481    0.855926
